In [0]:

%sql
-- Creating Valume directory
CREATE VOLUME IF NOT EXISTS my_catalog.default.raw_data;

In [0]:
import requests
import zipfile
import io
import os
import pyspark.sql.functions as F

In [0]:
# Is Executed importing the CSV Data of "Poland's House Prices" from Kaggle via API KEY and Kaggle API
volume_path = "/Volumes/my_catalog/default/raw_data"
dataset_url = "https://www.kaggle.com/api/v1/datasets/download/dawidcegielski/house-prices-in-poland"
kaggle_token = "Enter your Kaggle API Token"

print("Connect to Kaggle REST API")
headers = {"Authorization": f"Bearer {kaggle_token}"}
response = requests.get(dataset_url, headers=headers, stream=True)

if response.status_code == 200:
    print("Data is downloaded")
    print("Started unziping")
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for file_name in z.namelist():
            target_path = os.path.join(volume_path, file_name)
            with open(target_path, "wb") as f:
                f.write(z.read(file_name))
            print(f"File: {file_name} copleated")
            
    print(f"\nDone! Date storeging in: {os.listdir(volume_path)}")
else:
    print(f"Download error: {response.status_code}")
    print(response.text)



In [0]:
# Is Executed a transformation RAW_DATA to Bronze Table
file_path = "/Volumes/my_catalog/default/raw_data/Houses.csv"
df = spark.read.csv(file_path, header=True)
df.write.format("delta").mode("overwrite").saveAsTable("my_catalog.default.PolHousePrices_Bronze")

In [0]:
file_path = "my_catalog.default.PolHousePrices_Bronze"
df = spark.read.table(file_path)
display(df.limit(10))